# Map cells across data

In [1]:
import os
os.chdir('/nfs/users/nfs_v/vm11/projects/hdca_v2_reprocessing/03_link_reprocessed_to_author')

In [56]:
from datetime import date
import pandas as pd
import numpy as np
import sys
sys.path.append("..")
import utils.fetch_dbs
import utils.bcutils
import json
import mysql.connector
from pathlib import Path

In [3]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s : %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


In [5]:
storage_path = '/lustre/scratch124/cellgen/haniffa/users/ar32/fresh_public_object_curations/output_files/'
publication_id = 'vento_tormo_2018_placenta_decidua'
file_path = Path(storage_path) / publication_id / f'{publication_id}_scRNA_main.h5ad'

# example author object

In [31]:
logger.info(f"Reading data from {file_path}")
adata = sc.read_h5ad(file_path, backed = 'r')
adata.obs['publication_id'] = publication_id

2026-06-03 11:28:22 INFO : Reading data from /lustre/scratch124/cellgen/haniffa/users/ar32/fresh_public_object_curations/output_files/vento_tormo_2018_placenta_decidua/vento_tormo_2018_placenta_decidua_scRNA_main.h5ad


In [32]:
logger.info(f"Retain author obs names")
adata.obs['author_obs_names'] = adata.obs.index # Retain original cell-barcode names
adata.obs['author_obs_names'] = adata.obs['author_obs_names'].astype(str) # Ensure they are strings

2026-06-03 11:28:24 INFO : Retain author obs names


In [33]:
adata.obs['obs_unit_id'].unique()

['FCA7167219', 'FCA7167221', 'FCA7167222', 'FCA7167223', 'FCA7167224', ..., 'FCA7474065', 'FCA7474068', 'FCA7511881', 'FCA7511882', 'FCA7511884']
Length: 25
Categories (25, object): ['FCA7167219', 'FCA7167221', 'FCA7167222', 'FCA7167223', ..., 'FCA7474068', 'FCA7511881', 'FCA7511882', 'FCA7511884']

In [ ]:
# E-MTAB metadata (SDRF)
import re

def sdrf_col_to_snake(name: str) -> str:
    s = name.strip().lower()
    s = re.sub(r"[\[\]]", "_", s)
    s = re.sub(r"[\s\.]+", "_", s)
    s = re.sub(r"[^a-z0-9_]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")

lib_meta = pd.read_csv('../datasets_metadata/E-MTAB-6701.csv', index_col=0)
lib_meta.columns = [sdrf_col_to_snake(c) for c in lib_meta.columns]
lib_meta.head()


,Unnamed: 0,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[individual],Characteristics[sex],Characteristics[developmental stage],Characteristics[organism part],Characteristics[disease],...,Comment[cDNA read offset],Comment[cDNA read size],Comment[sample barcode read],Comment[sample barcode offset],Comment[sample barcode size],Comment[read1 file],Comment[read2 file],Comment[index1 file],Factor Value[organism part],Factor Value[FACS marker]
0,0,FCA7167219,ERS2657608,SAMEA4837741,Homo sapiens,D6,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
1,1,FCA7167221,ERS2657610,SAMEA4837743,Homo sapiens,D7,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167221_R1.fq.gz,FCA7167221_R2.fq.gz,FCA7167221_I1.fq.gz,decidua,CD45+
2,2,FCA7167222,ERS2657611,SAMEA4837744,Homo sapiens,D7,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167222_R1.fq.gz,FCA7167222_R2.fq.gz,FCA7167222_I1.fq.gz,decidua,CD45-
3,3,FCA7167223,ERS2657612,SAMEA4837745,Homo sapiens,D6,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167223_R1.fq.gz,FCA7167223_R2.fq.gz,FCA7167223_I1.fq.gz,decidua,CD45+
4,4,FCA7167224,ERS2657613,SAMEA4837746,Homo sapiens,D6,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167224_R1.fq.gz,FCA7167224_R2.fq.gz,FCA7167224_I1.fq.gz,decidua,CD45-


,Unnamed: 0,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[individual],Characteristics[sex],Characteristics[developmental stage],Characteristics[organism part],Characteristics[disease],...,Comment[cDNA read offset],Comment[cDNA read size],Comment[sample barcode read],Comment[sample barcode offset],Comment[sample barcode size],Comment[read1 file],Comment[read2 file],Comment[index1 file],Factor Value[organism part],Factor Value[FACS marker]
0,0,FCA7167219,ERS2657608,SAMEA4837741,Homo sapiens,D6,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
1,1,FCA7167221,ERS2657610,SAMEA4837743,Homo sapiens,D7,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167221_R1.fq.gz,FCA7167221_R2.fq.gz,FCA7167221_I1.fq.gz,decidua,CD45+
2,2,FCA7167222,ERS2657611,SAMEA4837744,Homo sapiens,D7,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167222_R1.fq.gz,FCA7167222_R2.fq.gz,FCA7167222_I1.fq.gz,decidua,CD45-
3,3,FCA7167223,ERS2657612,SAMEA4837745,Homo sapiens,D6,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167223_R1.fq.gz,FCA7167223_R2.fq.gz,FCA7167223_I1.fq.gz,decidua,CD45+
4,4,FCA7167224,ERS2657613,SAMEA4837746,Homo sapiens,D6,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167224_R1.fq.gz,FCA7167224_R2.fq.gz,FCA7167224_I1.fq.gz,decidua,CD45-
5,5,FCA7167226,ERS2657614,SAMEA4837747,Homo sapiens,D7,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7167226_R1.fq.gz,FCA7167226_R2.fq.gz,FCA7167226_I1.fq.gz,decidua,CD45-
6,6,FCA7167230,ERS2657618,SAMEA4837751,Homo sapiens,D7,female,adult,blood,normal,...,0,98,index1,0,8,FCA7167230_R1.fq.gz,FCA7167230_R2.fq.gz,FCA7167230_I1.fq.gz,blood,PB
7,7,FCA7167231,ERS2657619,SAMEA4837752,Homo sapiens,D6,female,adult,blood,normal,...,0,98,index1,0,8,FCA7167231_R1.fq.gz,FCA7167231_R2.fq.gz,FCA7167231_I1.fq.gz,blood,PB
8,8,FCA7167232,ERS2657620,SAMEA4837753,Homo sapiens,D6,female,adult,blood,normal,...,0,98,index1,0,8,FCA7167232_R1.fq.gz,FCA7167232_R2.fq.gz,FCA7167232_I1.fq.gz,blood,PB
9,9,FCA7196218,ERS2657621,SAMEA4837754,Homo sapiens,D8,female,adult,decidua,normal,...,0,98,index1,0,8,FCA7196218_R1.fq.gz,FCA7196218_R2.fq.gz,FCA7196218_I1.fq.gz,decidua,CD45+


In [35]:
adata.obs

,Fetus,location,final_cluster,annotation,barcode,obs_unit_id,original_author_annotation_broad,original_author_annotation_fine,publication_id,author_obs_names
FCA7167219_AAACGGGAGCTAGTTC,D6,Decidua,2,dNK2,AAACGGGAGCTAGTTC,FCA7167219,dNK2,dNK2,vento_tormo_2018_placenta_decidua,FCA7167219_AAACGGGAGCTAGTTC
FCA7167219_AAACGGGCATTGGCGC,D6,Decidua,5,dNK1,AAACGGGCATTGGCGC,FCA7167219,dNK1,dNK1,vento_tormo_2018_placenta_decidua,FCA7167219_AAACGGGCATTGGCGC
FCA7167219_AAACGGGTCGCGATCG,D6,Decidua,8,Tcells,AAACGGGTCGCGATCG,FCA7167219,Tcells,Tcells,vento_tormo_2018_placenta_decidua,FCA7167219_AAACGGGTCGCGATCG
FCA7167219_AAAGATGAGCAATATG,D6,Decidua,10,Tcells,AAAGATGAGCAATATG,FCA7167219,Tcells,Tcells,vento_tormo_2018_placenta_decidua,FCA7167219_AAAGATGAGCAATATG
FCA7167219_AAAGATGAGTTCGCGC,D6,Decidua,5,dNK1,AAAGATGAGTTCGCGC,FCA7167219,dNK1,dNK1,vento_tormo_2018_placenta_decidua,FCA7167219_AAAGATGAGTTCGCGC
...,...,...,...,...,...,...,...,...,...,...
FCA7511884_TTTGGTTGTTAAAGAC,D12,Placenta,12,fFB1,TTTGGTTGTTAAAGAC,FCA7511884,fFB1,fFB1,vento_tormo_2018_placenta_decidua,FCA7511884_TTTGGTTGTTAAAGAC
FCA7511884_TTTGGTTTCCATGCTC,D12,Placenta,9,VCT,TTTGGTTTCCATGCTC,FCA7511884,VCT,VCT,vento_tormo_2018_placenta_decidua,FCA7511884_TTTGGTTTCCATGCTC
FCA7511884_TTTGTCACACCGAAAG,D12,Placenta,16,EVT,TTTGTCACACCGAAAG,FCA7511884,EVT,EVT,vento_tormo_2018_placenta_decidua,FCA7511884_TTTGTCACACCGAAAG
FCA7511884_TTTGTCATCCGAGCCA,D12,Placenta,12,fFB1,TTTGTCATCCGAGCCA,FCA7511884,fFB1,fFB1,vento_tormo_2018_placenta_decidua,FCA7511884_TTTGTCATCCGAGCCA


In [60]:
# merge adata.obs with sdrf on obs_unit_id <-> source_name, preserving cell-barcode index
merged = (
    adata.obs.reset_index()
    .merge(lib_meta, how='left', left_on='obs_unit_id', right_on='source_name')
    .set_index(adata.obs.index.name or 'index')
)
merged

,Fetus,location,final_cluster,annotation,barcode,obs_unit_id,original_author_annotation_broad,original_author_annotation_fine,publication_id,author_obs_names,...,Comment[cDNA read offset]_y,Comment[cDNA read size]_y,Comment[sample barcode read]_y,Comment[sample barcode offset]_y,Comment[sample barcode size]_y,Comment[read1 file]_y,Comment[read2 file]_y,Comment[index1 file]_y,Factor Value[organism part]_y,Factor Value[FACS marker]_y
index,,,,,,,,,,,,,,,,,,,,,
FCA7167219_AAACGGGAGCTAGTTC,D6,Decidua,2,dNK2,AAACGGGAGCTAGTTC,FCA7167219,dNK2,dNK2,vento_tormo_2018_placenta_decidua,FCA7167219_AAACGGGAGCTAGTTC,...,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
FCA7167219_AAACGGGCATTGGCGC,D6,Decidua,5,dNK1,AAACGGGCATTGGCGC,FCA7167219,dNK1,dNK1,vento_tormo_2018_placenta_decidua,FCA7167219_AAACGGGCATTGGCGC,...,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
FCA7167219_AAACGGGTCGCGATCG,D6,Decidua,8,Tcells,AAACGGGTCGCGATCG,FCA7167219,Tcells,Tcells,vento_tormo_2018_placenta_decidua,FCA7167219_AAACGGGTCGCGATCG,...,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
FCA7167219_AAAGATGAGCAATATG,D6,Decidua,10,Tcells,AAAGATGAGCAATATG,FCA7167219,Tcells,Tcells,vento_tormo_2018_placenta_decidua,FCA7167219_AAAGATGAGCAATATG,...,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
FCA7167219_AAAGATGAGTTCGCGC,D6,Decidua,5,dNK1,AAAGATGAGTTCGCGC,FCA7167219,dNK1,dNK1,vento_tormo_2018_placenta_decidua,FCA7167219_AAAGATGAGTTCGCGC,...,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FCA7511884_TTTGGTTGTTAAAGAC,D12,Placenta,12,fFB1,TTTGGTTGTTAAAGAC,FCA7511884,fFB1,fFB1,vento_tormo_2018_placenta_decidua,FCA7511884_TTTGGTTGTTAAAGAC,...,0,98,index1,0,8,FCA7511884_S1_L001_R1_001.fastq.gz,FCA7511884_S1_L001_R2_001.fastq.gz,FCA7511884_S1_L001_I1_001.fastq.gz,placenta,all
FCA7511884_TTTGGTTTCCATGCTC,D12,Placenta,9,VCT,TTTGGTTTCCATGCTC,FCA7511884,VCT,VCT,vento_tormo_2018_placenta_decidua,FCA7511884_TTTGGTTTCCATGCTC,...,0,98,index1,0,8,FCA7511884_S1_L001_R1_001.fastq.gz,FCA7511884_S1_L001_R2_001.fastq.gz,FCA7511884_S1_L001_I1_001.fastq.gz,placenta,all
FCA7511884_TTTGTCACACCGAAAG,D12,Placenta,16,EVT,TTTGTCACACCGAAAG,FCA7511884,EVT,EVT,vento_tormo_2018_placenta_decidua,FCA7511884_TTTGTCACACCGAAAG,...,0,98,index1,0,8,FCA7511884_S1_L001_R1_001.fastq.gz,FCA7511884_S1_L001_R2_001.fastq.gz,FCA7511884_S1_L001_I1_001.fastq.gz,placenta,all


In [37]:
logger.warning("Only proceed if the indices match")

if not adata.obs.index.equals(merged.index):
    logger.error("Index mismatch between adata.obs and merged.index")
else:
    logger.info("Index matches between adata.obs and merged.index")
    adata.obs = merged


2026-06-03 11:28:33 WARNING : Only proceed if the indices match
2026-06-03 11:28:33 INFO : Index matches between adata.obs and merged.index


In [24]:
adata

AnnData object with n_obs × n_vars = 64734 × 31764 backed at '/lustre/scratch124/cellgen/haniffa/users/ar32/fresh_public_object_curations/output_files/vento_tormo_2018_placenta_decidua/vento_tormo_2018_placenta_decidua_scRNA_main.h5ad'
    obs: 'Fetus', 'location', 'final_cluster', 'annotation', 'barcode', 'obs_unit_id', 'original_author_annotation_broad', 'original_author_annotation_fine', 'publication_id', 'author_obs_names', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[individual]', 'Characteristics[sex]', 'Characteristics[developmental stage]', 'Characteristics[organism part]', 'Characteristics[disease]', 'Characteristics[clinical information]', 'Characteristics[FACS marker]', 'Characteristics[run]', 'Comment[single cell isolation]', 'Comment[library construction]', 'Comment[input molecule]', 'Comment[primer]', 'Comment[end bias]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Extract Name', 'Co

In [25]:
adata.obs.publication_id

index
FCA7167219_AAACGGGAGCTAGTTC    vento_tormo_2018_placenta_decidua
FCA7167219_AAACGGGCATTGGCGC    vento_tormo_2018_placenta_decidua
FCA7167219_AAACGGGTCGCGATCG    vento_tormo_2018_placenta_decidua
FCA7167219_AAAGATGAGCAATATG    vento_tormo_2018_placenta_decidua
FCA7167219_AAAGATGAGTTCGCGC    vento_tormo_2018_placenta_decidua
                                             ...                
FCA7511884_TTTGGTTGTTAAAGAC    vento_tormo_2018_placenta_decidua
FCA7511884_TTTGGTTTCCATGCTC    vento_tormo_2018_placenta_decidua
FCA7511884_TTTGTCACACCGAAAG    vento_tormo_2018_placenta_decidua
FCA7511884_TTTGTCATCCGAGCCA    vento_tormo_2018_placenta_decidua
FCA7511884_TTTGTCATCTTGAGGT    vento_tormo_2018_placenta_decidua
Name: publication_id, Length: 64734, dtype: object

In [38]:
adata.obs['library_id'] = adata.obs['comment_ena_sample']

In [39]:
np.savetxt('library_ids.txt', adata.obs['library_id'].unique(), fmt='%s')

In [41]:
libraries = pd.read_csv('../libraries_lists/hdca_v2_libraries260529.csv')

In [42]:

libs = libraries[libraries.publication_id==publication_id]
libs.index = libs.library_id
libs

,publication_id,institute,source,dataset_acc,library_id,author_sample_id,sanger_10x_chemistry,irods_path,HDCA_version
library_id,,,,,,,,,
ERS2657608,vento_tormo_2018_placenta_decidua,Wellcome Sanger Institute,external,E-MTAB-6701,ERS2657608,FCA7167219,Chromium single cell,/archive/cellgeni/datasets/E-MTAB-6701/ERS2657608,v2
ERS2657610,vento_tormo_2018_placenta_decidua,Wellcome Sanger Institute,external,E-MTAB-6701,ERS2657610,FCA7167221,Chromium single cell,/archive/cellgeni/datasets/E-MTAB-6701/ERS2657610,v2
ERS2657611,vento_tormo_2018_placenta_decidua,Wellcome Sanger Institute,external,E-MTAB-6701,ERS2657611,FCA7167222,Chromium single cell,/archive/cellgeni/datasets/E-MTAB-6701/ERS2657611,v2
ERS2657612,vento_tormo_2018_placenta_decidua,Wellcome Sanger Institute,external,E-MTAB-6701,ERS2657612,FCA7167223,Chromium single cell,/archive/cellgeni/datasets/E-MTAB-6701/ERS2657612,v2
ERS2657613,vento_tormo_2018_placenta_decidua,Wellcome Sanger Institute,external,E-MTAB-6701,ERS2657613,FCA7167224,Chromium single cell,/archive/cellgeni/datasets/E-MTAB-6701/ERS2657613,v2
ERS2657614,vento_tormo_2018_placenta_decidua,Wellcome Sanger Institute,external,E-MTAB-6701,ERS2657614,FCA7167226,Chromium single cell,/archive/cellgeni/datasets/E-MTAB-6701/ERS2657614,v2
ERS2657618,vento_tormo_2018_placenta_decidua,Wellcome Sanger Institute,external,E-MTAB-6701,ERS2657618,FCA7167230,Chromium single cell,/archive/cellgeni/datasets/E-MTAB-6701/ERS2657618,v2
ERS2657619,vento_tormo_2018_placenta_decidua,Wellcome Sanger Institute,external,E-MTAB-6701,ERS2657619,FCA7167231,Chromium single cell,/archive/cellgeni/datasets/E-MTAB-6701/ERS2657619,v2
ERS2657620,vento_tormo_2018_placenta_decidua,Wellcome Sanger Institute,external,E-MTAB-6701,ERS2657620,FCA7167232,Chromium single cell,/archive/cellgeni/datasets/E-MTAB-6701/ERS2657620,v2


In [43]:

# load reprocessed barcodes from irods
rbc = utils.fetch_dbs.read_gzipped_irods_files_as_set(libs.irods_path+'/output/GeneFull/filtered/barcodes.tsv.gz')
rbc = dict(zip(libs.library_id,rbc))



2026-06-03 11:33:02 INFO : Native authorization validated (in legacy auth).
2026-06-03 11:33:02 INFO : Native authorization validated (in legacy auth).


In [46]:
len(rbc)

30

In [51]:
libs_in_adata = adata.obs.library_id.unique()

In [ ]:
# Libs reprocessed but not in author object
# Decision - Disregard these
libs[~libs.library_id.isin(libs_in_adata)].author_sample_id

library_id
ERS2657633    FCA7474066
ERS2657636    FCA7474069
ERS2657639    FCA7511883
ERS2657641    FCA7511885
ERS2657642    FCA7511886
Name: author_sample_id, dtype: object

In [63]:
aobs = adata.obs

In [75]:
abc = aobs.groupby('library_id')['barcode'].apply(set).to_dict()

In [ ]:
# get author barcodes
abc = aobs.groupby('library_id')['barcode'].apply(set).to_dict()
overlap_matrix, jaccard_matrix, keys_a, keys_r, matches  = utils.bcutils.compute_overlap_matrices(abc,rbc)
matches['author_sample_id'] = libs.loc[matches.key_b,'author_sample_id'].tolist()
matches['dataset_acc'] = libs.loc[matches.key_b,'dataset_acc'].tolist()
matches['cells_lost_in_reprocessing'] = matches['size_a'] - matches['overlap']
matches['fraction_lost_in_reprocessing'] = matches['cells_lost_in_reprocessing'] / matches['size_a']

,key_a,key_b,size_a,size_b,overlap,jaccard,szymkiewicz–simpson,author_sample_id,dataset_acc
0,ERS2657624,ERS2657624,6266,8948,6257,0.698560,0.998564,FCA7196224,E-MTAB-6701
1,ERS2657621,ERS2657621,5277,8720,5263,0.602588,0.997347,FCA7196218,E-MTAB-6701
2,ERS2657614,ERS2657614,4916,7085,4902,0.690520,0.997152,FCA7167226,E-MTAB-6701
3,ERS2657623,ERS2657623,4656,7275,4654,0.639549,0.999570,FCA7196220,E-MTAB-6701
4,ERS2657628,ERS2657628,4538,4612,4538,0.983955,1.000000,FCA7196231,E-MTAB-6701
5,ERS2657612,ERS2657612,4167,8150,4149,0.507958,0.995680,FCA7167223,E-MTAB-6701
6,ERS2657631,ERS2657631,3801,4874,3801,0.779852,1.000000,FCA7474064,E-MTAB-6701
7,ERS2657632,ERS2657632,3672,4785,3672,0.767398,1.000000,FCA7474065,E-MTAB-6701
8,ERS2657622,ERS2657622,3120,3437,3116,0.905551,0.998718,FCA7196219,E-MTAB-6701
9,ERS2657627,ERS2657627,3004,3120,3004,0.962821,1.000000,FCA7196229,E-MTAB-6701


In [78]:
matches['cells_lost_in_reprocessing'] = matches['size_a'] - matches['overlap']
matches['fraction_lost_in_reprocessing'] = matches['cells_lost_in_reprocessing'] / matches['size_a']

In [79]:
matches

,key_a,key_b,size_a,size_b,overlap,jaccard,szymkiewicz–simpson,author_sample_id,dataset_acc,cells_lost_in_reprocessing,fraction_lost_in_reprocessing
0,ERS2657624,ERS2657624,6266,8948,6257,0.698560,0.998564,FCA7196224,E-MTAB-6701,9,0.001436
1,ERS2657621,ERS2657621,5277,8720,5263,0.602588,0.997347,FCA7196218,E-MTAB-6701,14,0.002653
2,ERS2657614,ERS2657614,4916,7085,4902,0.690520,0.997152,FCA7167226,E-MTAB-6701,14,0.002848
3,ERS2657623,ERS2657623,4656,7275,4654,0.639549,0.999570,FCA7196220,E-MTAB-6701,2,0.000430
4,ERS2657628,ERS2657628,4538,4612,4538,0.983955,1.000000,FCA7196231,E-MTAB-6701,0,0.000000
5,ERS2657612,ERS2657612,4167,8150,4149,0.507958,0.995680,FCA7167223,E-MTAB-6701,18,0.004320
6,ERS2657631,ERS2657631,3801,4874,3801,0.779852,1.000000,FCA7474064,E-MTAB-6701,0,0.000000
7,ERS2657632,ERS2657632,3672,4785,3672,0.767398,1.000000,FCA7474065,E-MTAB-6701,0,0.000000
8,ERS2657622,ERS2657622,3120,3437,3116,0.905551,0.998718,FCA7196219,E-MTAB-6701,4,0.001282
9,ERS2657627,ERS2657627,3004,3120,3004,0.962821,1.000000,FCA7196229,E-MTAB-6701,0,0.000000
